# Lab 6 — Deep Learning for NLP (Self-contained)
**Run the next two cells first.** This version installs missing packages and recreates the Yelp dataset automatically.

In [ ]:
# SETUP CELL — run this first
# Installs all required packages in the current Jupyter environment.
import sys, subprocess, importlib.util

required = {
    "torch": "torch",
    "gensim": "gensim",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib"
}

for module_name, package_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("✅ Required packages are installed.")
print("If torch was just installed and the next cell still cannot import it, restart the kernel once and run all cells again.")


In [ ]:
# EMBEDDED DATASET — no separate .txt file is required
import base64, os

_dataset_b64 = '''...'''
dataset_path = "07-yelp-dataset.txt"

if not os.path.exists(dataset_path):
    with open(dataset_path, "wb") as f:
        f.write(base64.b64decode(_dataset_b64))

print(f"✅ Dataset ready: {dataset_path}")


# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [ ]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

In [ ]:
print(df["label"].value_counts())

## 2. Preprocessing the Dataset

In [ ]:
import re

def preprocess_text(text):
    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply preprocessing to all reviews
df["review"] = df["review"].apply(preprocess_text)
df.head()


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [ ]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [ ]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # Reset old gradients before each update
    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [ ]:
from gensim.models import Word2Vec
import numpy as np

# 1. Tokenize the reviews (the text was already cleaned above)
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

# 2. Train Word2Vec ONLY on the training reviews
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=100
)

# 3. Convert one review into one vector by averaging its word vectors
def review_to_vector(tokens, model, vector_size=200):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

X_train_w2v = np.vstack([review_to_vector(tokens, w2v_model) for tokens in train_tokens])
X_test_w2v = np.vstack([review_to_vector(tokens, w2v_model) for tokens in test_tokens])

# 4. Convert the Word2Vec review vectors to PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)
y_train_w2v_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

print("Word2Vec vocabulary size:", len(w2v_model.wv))
print("Training Word2Vec shape:", X_train_w2v_tensor.shape)
print("Testing Word2Vec shape:", X_test_w2v_tensor.shape)


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [ ]:
# Reproducibility
torch.manual_seed(42)

# Input size is 200 because vector_size=200 in Word2Vec
w2v_input_size = X_train_w2v_tensor.shape[1]

w2v_nn_model = nn.Sequential(
    nn.Linear(w2v_input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

loss_function_w2v = nn.BCEWithLogitsLoss()
optimizer_w2v = torch.optim.Adam(w2v_nn_model.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    w2v_nn_model.train()
    optimizer_w2v.zero_grad()

    outputs = w2v_nn_model(X_train_w2v_tensor)
    loss = loss_function_w2v(outputs, y_train_w2v_tensor)

    loss.backward()
    optimizer_w2v.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

# Evaluate on the test set
w2v_nn_model.eval()
with torch.no_grad():
    test_logits = w2v_nn_model(X_test_w2v_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    w2v_predictions = (test_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Word2Vec Neural Network Test Accuracy: {w2v_accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, w2v_predictions))
